# 02 — Environmental Parameters

`POST /v1/env_params` returns thermal-comfort, air-quality, and solar-irradiance metrics for a point.

**Plan:** available on both tiers — Basic is limited to 3 parameters per request.

We'll request a single hour at a point, then plot the time-series parameters returned.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv; load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
client = FortyGuardClient()

In [ ]:
response = client.environmental_parameters(
    latitude=40.7128,
    longitude=-74.0060,
    temperature=32.5,
    start_date='2024-07-15',
    start_time='09:00',
    end_time='17:00',
    filter_type=2,   # range of hours
)

result = response['result']
print('Metadata:', result.get('metadata', {}))

In [ ]:
import pandas as pd

location = result['locations'][0]
timestamps = result['metadata'].get('timestamps', [])
params = location.get('parameters', {})

df = pd.DataFrame({k: v for k, v in params.items() if isinstance(v, list) and len(v) == len(timestamps)})
df.insert(0, 'timestamp', pd.to_datetime(timestamps))
df.set_index('timestamp', inplace=True)
df.head()

In [ ]:
import matplotlib.pyplot as plt

to_plot = [c for c in ['heat_index_celsius', 'apparent_temperature_celsius', 'wet_bulb_temperature_celsius', 'relative_humidity_percent'] if c in df.columns]
if to_plot:
    df[to_plot].plot(figsize=(10, 4), marker='o')
    plt.title('Thermal comfort parameters over the requested window')
    plt.ylabel('value'); plt.tight_layout(); plt.show()

In [ ]:
solar = location.get('solar_irradiance', {}).get('clear_sky', {})
print('Clear-sky solar irradiance (W/m²):')
for comp in ('ghi', 'dni', 'dhi'):
    print(f'  {comp.upper():>5}: {solar.get(comp)}')